# FloodA5 notebook 5 — performance: CPU threads, GPU, Float64 and Float32

How fast FloodA5's time stepping runs on a machine, and what the faster
options cost in accuracy. Three configurations are compared throughout:

| Configuration | Command line | What it is |
|---|---|---|
| **CPU threads** | (default) | the reference path, Float64 |
| **GPU Float64** | `--backend gpu` | the same arithmetic on an NVIDIA GPU (CUDA) |
| **GPU Float32** | `--backend gpu --precision float32` | fluxes and momentum in single precision (0.6.0) |

`--precision float32` (standard flow and SWE; `surfacewater/precision.jl`)
keeps every cell's **volume in Float64** and exchanges water between two
cells as one number added to one and subtracted from the other, so mass is
conserved exactly as in a Float64 run. Water levels are never formed as
bed + depth and then differenced — at a bed elevation of 500 m a Float32
level resolves only 3·10⁻⁵ m — but taken relative to a local datum (the
face's sill, or the neighbouring cell's bed), from offsets that are
differenced in Float64 once, before the run. Most GPUs do double-precision
arithmetic far more slowly than single precision (1/64 of the rate on many
workstation and consumer cards), which is what Float32 is for.

The notebook

1. identifies **this machine** and lists the machines that already have
   results (§1);
2. **runs the benchmarks this machine has no results for** — nothing, if it
   has them all — and saves them to `benchmarks/results/<machine>.json` (§2);
3. shows the **step throughput** on synthetic grids of 65 thousand to 4
   million cells (§3) and how the CPU step **scales with threads** (§4);
4. shows the **Carlisle** test runs: wall time of whole simulations from the
   command line (§5);
5. shows the **accuracy** side of the trade: how far GPU and Float32 results
   are from the CPU Float64 run (§6);
6. puts the **machines side by side** (§7).

**To add a machine:** check out the repository there, open this notebook and
run it. The results file it writes is small and is meant to be committed;
after a `git pull` every copy of the notebook shows every machine. The same
machine under another operating system counts as another machine (its file
is named after the host name and the OS).

## 0. Set-up

The cell below finds the repository from the Julia project the kernel was
started with and loads `BS` (`benchmarks/BenchSuite.jl`, the benchmark
runner), `FA` (`analysis/FloodA5Analysis.jl`) and `FP`
(`visualisation/FloodA5Plots.jl`, the figure style). The kernel needs the
full project (`--project=<repository>`): the GPU runs use CUDA.jl, which the
minimal `ci` project does not have.

- `RUN_BENCHMARKS = false` only shows results that already exist.
- `FORCE = true` runs the sections in `SECTIONS` again on this machine and
  replaces its results (after a code change, say).
- `LARGE = false` leaves out the 5 m Carlisle mesh (the longest runs).
- `LABEL` names this machine in the results; empty, it is the host name.

Use a machine that is doing nothing else: every number here is a wall time.
The full suite takes about an hour and a half on a 16-thread workstation with
a GPU (§3 ten minutes, §4 fifteen, §5 the rest); each measurement is a
separate Julia process, and the cell prints each one as it finishes.

In [ ]:
# The kernel starts Julia with --project=<repository>: walk up from that
# project to the directory holding FloodModel.jl.
const REPO = let d = dirname(something(Base.active_project(), joinpath(pwd(), "Project.toml")))
    while !isfile(joinpath(d, "FloodModel.jl"))
        dirname(d) == d && error("no FloodA5 repository above $(Base.active_project())")
        d = dirname(d)
    end
    d
end
ENV["FLOODA5_NO_VIS"] = "1"            # the runs need no viewer

include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));     const FA = FloodA5Analysis
include(joinpath(REPO, "benchmarks", "BenchSuite.jl"));        const BS = BenchSuite
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl"));   const FP = FloodA5Plots
using CairoMakie, Printf, Statistics
FP.activate!()                         # the FloodA5 figure style

const RUN_BENCHMARKS = true            # false: only show existing results
const FORCE    = false                 # true: run SECTIONS again on this machine
const SECTIONS = [:step, :threads, :carlisle]
const LARGE    = true                  # include the 5 m Carlisle mesh (~580 000 cells)
const LABEL    = ""                    # this machine's name in the results ("" = host name)

println("repository : ", REPO)
println("version    : ", strip(read(joinpath(REPO, "VERSION"), String)))
println("results    : ", BS.RESULTS_DIR)

Helpers: `mdtable` shows a table (as HTML), `f` formats a number; `sel` and
`pick` take rows out of a results section; the three configurations have
fixed colours, and each machine its own marker and line style.

In [ ]:
"An HTML table from a header and rows of already formatted values."
function mdtable(header, rows)
    esc(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    cell(tag, x) = "<$tag style=\"text-align:right\">" * esc(x) * "</$tag>"
    s = "<table><thead><tr>" * join(cell("th", h) for h in header) * "</tr></thead><tbody>"
    for r in rows
        s *= "<tr>" * join(cell("td", v) for v in r) * "</tr>"
    end
    return HTML(s * "</tbody></table>")
end
"`x` to `d` decimals, or as 1.2e-05 if it would round to zero; — if there is no value."
f(x, d = 3) = (x === nothing || (x isa Real && isnan(x))) ? "—" : !isfinite(x) ? string(x) :
              (x != 0 && abs(x) < 0.5 * 10.0^-d) ? @sprintf("%.1e", x) : string(round(x; digits = d))

"Rows of a section with the given field values; `pick`: the one such row, or nothing."
sel(rows; kw...) = [r for r in rows if all(get(r, String(k), nothing) == v for (k, v) in kw)]
pick(rows; kw...) = (s = sel(rows; kw...); isempty(s) ? nothing : s[end])
"Field `k` of a row as a number (NaN if the row or the field is missing)."
val(r, k) = (r === nothing || get(r, k, nothing) === nothing) ? NaN : Float64(r[k])

const CONFIGS = ["cpu", "gpu64", "gpu32"]
const CONFIG_LABEL = Dict("cpu" => "CPU threads", "gpu64" => "GPU Float64", "gpu32" => "GPU Float32")
const CONFIG_COLOR = Dict("cpu" => FP.SERIES[1], "gpu64" => FP.SERIES[2], "gpu32" => FP.SERIES[3])
const SOLVER_LABEL = Dict("standard" => "standard flow (local-inertial)", "swe" => "SWE reference (order 2)")
marker(i) = FP.MARKERS[mod1(i, length(FP.MARKERS))]
linestyle(i) = FP.LINESTYLES[mod1(i, length(FP.LINESTYLES))]
name(r) = r["machine"]["label"] * " (" * r["machine"]["os"] * ")"
nothing

## 1. This machine, and the machines with results

A machine is identified by its host name and operating system. The GPU is
the one `--backend gpu` would use, as CUDA.jl reports it (asked in a separate
process, which takes half a minute the first time).

In [ ]:
me = BS.machine(; label = LABEL)
println("this machine : ", me["id"])
println("  CPU        : ", me["cpu"], " — ", me["cores"], " cores, ", me["threads"], " threads, ", me["memory_gb"], " GB")
println("  GPU        : ", me["gpu"] == "" ? "none usable (CUDA.jl missing from the project, or no functional GPU)" :
                           "$(me["gpu"]), $(me["gpu_memory_gb"]) GB (CUDA $(me["cuda_runtime"]), driver $(me["gpu_driver"]))")
println("  Julia      : ", me["julia"], " on ", me["os_version"])

function machine_table(results)
    mdtable(["", "machine", "CPU", "cores / threads", "memory (GB)", "GPU", "GPU memory (GB)", "FloodA5", "sections"],
            [[r["machine"]["id"] == me["id"] ? "this machine →" : "", name(r), r["machine"]["cpu"],
              "$(r["machine"]["cores"]) / $(r["machine"]["threads"])", r["machine"]["memory_gb"],
              r["machine"]["gpu"] == "" ? "—" : r["machine"]["gpu"],
              r["machine"]["gpu"] == "" ? "—" : r["machine"]["gpu_memory_gb"],
              join(unique(s["flooda5_version"] for s in values(r["sections"])), ", "),
              join(("$k ($(first(s["date"], 10)))" for (k, s) in sort(collect(r["sections"]); by = first)), ", ")]
             for r in results])
end
before = BS.load_results()
println("\n", length(before), " machine(s) with results",
        any(r -> r["machine"]["id"] == me["id"], before) ? ", this one among them" : "; none for this machine yet")
machine_table(before)

## 2. Run what is missing

`BS.run_suite` runs the sections this machine has no results for, in order,
and saves after each. Run from the command line it is
`julia --project=. benchmarks/run.jl` (`--force`, `--sections …`,
`--no-large`, `--label …`; `--quick` is a fifteen-minute smoke test of the whole
chain whose numbers are not benchmarks).

- **step** — `test/benchmark_backend.jl`: square Cartesian grids of 256² to
  2048² cells, every cell wet (a dam break over a sloping bed, open
  boundaries), 20 steps timed after 2 warm-up steps, so compilation is not
  counted. Standard flow with the default scheme, and the SWE reference at
  order 2.
- **threads** — the CPU step on the 1024² grid at 1, 2, 4, … threads.
- **carlisle** — the Carlisle test runs from the command line (§5). The A5
  res-18 and 20 m Cartesian meshes are the `carlisle` case's
  (`julia --project=. cases/run.jl prep carlisle`, which needs the data in
  `test/carlisle/`); the 5 m mesh is built here from the LiDAR DEM. Meshes
  that cannot be found or built are left out, with a message.

In [ ]:
mine = RUN_BENCHMARKS ? BS.run_suite(; sections = SECTIONS, force = FORCE, label = LABEL, large = LARGE) :
                        BS.load_mine(; label = LABEL)
results = BS.load_results()
machine_table(results)

## 3. Step throughput

**Million cell-steps per second** (Mc·s/s): cells × steps / wall time. A run
of N cells for S steps takes N·S / throughput seconds, plus output.

What to look for:

- **GPU against CPU.** A GPU step is a fixed number of kernel launches
  (about 20 for standard flow) whatever the grid size, so small grids are
  dominated by launch latency and the GPU only pays above roughly 10⁵ cells.
  The CPU rate falls a little as the grid outgrows the caches; the GPU rate
  rises until the device is saturated.
- **Float32 against Float64 on the GPU.** Close to 2× where the kernels are
  limited by double-precision arithmetic or by memory traffic (the arrays
  are half the size); less on small grids, where launches dominate.
- All cells are wet here. On a mostly dry domain the CPU skips dry edges
  (the active-edge set) and the GPU does not, which moves the crossover up.

In [ ]:
function step_table(r)
    rws = BS.rows(r, :step)
    out = []
    for solver in ("standard", "swe"), n in sort(unique(x["n"] for x in sel(rws; solver)))
        c, g64, g32 = (pick(rws; solver, n, config) for config in CONFIGS)
        push!(out, [solver, "$(n) × $(n)", something(c, g64, g32)["cells"],
                    f(val(c, "mcs"), 1), f(val(g64, "mcs"), 1), f(val(g32, "mcs"), 1),
                    f(val(g64, "mcs") / val(c, "mcs"), 2), f(val(g32, "mcs") / val(c, "mcs"), 2),
                    f(val(g32, "mcs") / val(g64, "mcs"), 2)])
    end
    mdtable(["solver", "grid", "cells", "CPU (Mc·s/s)", "GPU Float64", "GPU Float32",
             "GPU64 / CPU", "GPU32 / CPU", "GPU32 / GPU64"], out)
end
println(name(mine), ": step throughput, ", get(get(mine["sections"], "step", Dict()), "date", "not run"))
step_table(mine)

In [ ]:
function step_figure(results)
    fig = Figure(size = (1050, 470))
    axs = Axis[]
    for (k, solver) in enumerate(("standard", "swe"))
        ax = Axis(fig[1, k]; xscale = log10, yscale = log10, xlabel = "cells",
                  ylabel = k == 1 ? "million cell-steps per second" : "", title = SOLVER_LABEL[solver])
        for (i, r) in enumerate(results), config in CONFIGS
            rws = sort(sel(BS.rows(r, :step); solver, config); by = x -> x["cells"])
            isempty(rws) && continue
            scatterlines!(ax, [x["cells"] for x in rws], [x["mcs"] for x in rws]; color = CONFIG_COLOR[config],
                          marker = marker(i), linestyle = linestyle(i), label = "$(CONFIG_LABEL[config]) — $(name(r))")
        end
        push!(axs, ax)
    end
    linkyaxes!(axs...)
    Legend(fig[2, 1:2], axs[1]; orientation = :horizontal, nbanks = max(1, length(results)), framevisible = false)
    fig
end
step_figure(results)

The same data as ratios: what Float32 gains over Float64 on each machine's
GPU, and the GPU (Float32) over that machine's CPU threads.

In [ ]:
function ratio_figure(results)
    fig = Figure(size = (1050, 400))
    for (k, (num, den, title)) in enumerate((("gpu32", "gpu64", "GPU Float32 / GPU Float64"),
                                             ("gpu32", "cpu", "GPU Float32 / CPU threads")))
        ax = Axis(fig[1, k]; xscale = log10, yscale = k == 2 ? log10 : identity, xlabel = "cells",
                  ylabel = "throughput ratio", title = title)
        hlines!(ax, [1.0]; color = (:black, 0.4), linewidth = 1)
        for (i, r) in enumerate(results), (j, solver) in enumerate(("standard", "swe"))
            rws = BS.rows(r, :step)
            ns = sort(unique(x["n"] for x in sel(rws; solver)))
            pts = [(val(pick(rws; solver, n, config = num), "cells"),
                    val(pick(rws; solver, n, config = num), "mcs") / val(pick(rws; solver, n, config = den), "mcs")) for n in ns]
            pts = [p for p in pts if isfinite(p[2])]
            isempty(pts) && continue
            scatterlines!(ax, first.(pts), last.(pts); color = FP.SERIES[3 + j], marker = marker(i),
                          linestyle = linestyle(i), label = "$solver — $(name(r))")
        end
        k == 2 && axislegend(ax; position = :lt, framevisible = false)
    end
    fig
end
ratio_figure(results)

## 4. Thread scaling of the CPU step

The CPU step on the 1024 × 1024 grid (about a million cells, all wet) at 1,
2, 4, … threads, one process per count. The step loops run on physical
cores (Polyester `@batch per=core`), so thread counts above the number of
cores add nothing. The solver is limited by memory traffic well before it
runs out of cores: expect the efficiency (speed-up ÷ threads) to fall as
threads are added, by more on machines with few memory channels.

In [ ]:
function threads_table(r)
    rws = BS.rows(r, :threads); out = []
    for solver in ("standard", "swe")
        s = sort(sel(rws; solver); by = x -> x["threads"])
        isempty(s) && continue
        base = s[1]["mcs"] / s[1]["threads"]
        for x in s
            push!(out, [solver, x["threads"], f(x["mcs"], 2), f(x["mcs"] / s[1]["mcs"], 2),
                        f(100 * x["mcs"] / (base * x["threads"]), 0) * "%"])
        end
    end
    mdtable(["solver", "threads", "Mc·s/s", "speed-up", "efficiency"], out)
end
function threads_figure(results)
    fig = Figure(size = (1050, 420))
    for (k, solver) in enumerate(("standard", "swe"))
        ax = Axis(fig[1, k]; xscale = log2, yscale = log2, xlabel = "threads",
                  ylabel = k == 1 ? "million cell-steps per second" : "", title = SOLVER_LABEL[solver])
        for (i, r) in enumerate(results)
            s = sort(sel(BS.rows(r, :threads); solver); by = x -> x["threads"])
            isempty(s) && continue
            t = [x["threads"] for x in s]
            lines!(ax, t, s[1]["mcs"] .* t ./ t[1]; color = (FP.SERIES[i], 0.35), linestyle = :dot)   # ideal
            scatterlines!(ax, t, [x["mcs"] for x in s]; color = FP.SERIES[mod1(i, length(FP.SERIES))],
                          marker = marker(i), label = name(r))
        end
        k == 1 && axislegend(ax; position = :lt, framevisible = false)
    end
    fig
end
println(name(mine), " (", mine["machine"]["cores"], " cores): dotted lines in the figure are ideal scaling")
display(threads_table(mine))
threads_figure(results)

## 5. Carlisle: whole runs from the command line

The step benchmark leaves out everything that is not the step: the time-step
calculation, sources, output, log lines, and on a GPU the copies back to the
host. These are complete runs of `FloodModel.jl`, timed by the model itself
(the `Simulation` line of its timing summary, `wall_simulation_s` in the
output file), with the Carlisle test forcing of `cases/registry/carlisle.jl`:
the three river inflows plus 10 mm/h of uniform rain, so every cell is wet
within minutes.

| Mesh | Cells | Simulated | Runs |
|---|---|---|---|
| A5 res 18 (`a5_res18`) | 29 902 | 6 h | standard, SWE × CPU, GPU Float64, GPU Float32 |
| Cartesian 20 m (`cart20`) | 36 414 | 6 h | the same |
| Cartesian 5 m (`cart5`) | about 580 000 | 1 h | standard × CPU, GPU Float64, GPU Float32; SWE on the GPU only |

The two 6 h meshes are small for a GPU — a 30 000-cell step is mostly launch
latency — and are here because they are the project's reference runs. The
5 m mesh is the size at which a GPU is worth having. The wall time of a GPU
run includes compiling its kernels (the *first 5 steps* column: tens of
seconds, once per process), which matters for runs this short and is
negligible for long ones; **× real time** is simulated time over wall time.

The commands, exactly as run on this machine:

In [ ]:
for r in BS.rows(mine, :carlisle)
    println(r["mesh"], ", ", r["model"], ", ", r["config"], ":\n  ", r["command"], "\n")
end

In [ ]:
function carlisle_table(r)
    rws = BS.rows(r, :carlisle); out = []
    for x in rws
        ref = pick(rws; mesh = x["mesh"], model = x["model"], config = get(x, "ref_config", x["config"]))
        push!(out, [x["mesh"], x["model"], CONFIG_LABEL[x["config"]], x["cells"], x["threads"],
                    f(x["sim_s"] / 3600, 1), x["steps"], f(x["wall_s"], 1), f(x["first5_s"], 1),
                    f(x["speed"], 0), haskey(x, "ref_config") ? f(val(ref, "wall_s") / x["wall_s"], 2) *
                                                                " × " * CONFIG_LABEL[x["ref_config"]] : "(reference)"])
    end
    mdtable(["mesh", "model", "configuration", "cells", "threads", "simulated (h)", "steps", "wall (s)",
             "first 5 steps (s)", "× real time", "speed"], out)
end
println(name(mine), ": Carlisle runs, ", get(get(mine["sections"], "carlisle", Dict()), "date", "not run"))
carlisle_table(mine)

In [ ]:
# Simulated time per wall time for every run: one column per mesh and model,
# colour = configuration, marker = machine.
function carlisle_figure(results)
    groups = [(m, s) for m in ("a5_res18", "cart20", "cart5") for s in ("standard", "swe")]
    fig = Figure(size = (1050, 470))
    ax = Axis(fig[1, 1]; yscale = log10, ylabel = "simulated time / wall time",
              xticks = (1:length(groups), ["$m\n$s" for (m, s) in groups]))
    for (i, r) in enumerate(results), (g, (mesh, model)) in enumerate(groups), (c, config) in enumerate(CONFIGS)
        x = pick(BS.rows(r, :carlisle); mesh, model, config)
        x === nothing && continue
        scatter!(ax, [g + 0.22 * (c - 2)], [x["speed"]]; color = CONFIG_COLOR[config], marker = marker(i),
                 markersize = 14, label = "$(CONFIG_LABEL[config]) — $(name(r))")
    end
    Legend(fig[2, 1], ax; orientation = :horizontal, nbanks = max(1, length(results)), framevisible = false,
           unique = true)
    fig
end
carlisle_figure(results)

## 6. Accuracy: what the faster configurations cost

Two different things move a result away from the CPU Float64 run, and the
tables below separate them by showing **GPU Float64** next to **GPU
Float32**, both against the CPU run.

- **Round-off amplification.** The local-inertial solver amplifies
  round-off-level differences where cells wet and dry, most of all on A5
  (`docs/HYDRAULICS.md` §15.5). A GPU sums in a different order from the CPU,
  so even in Float64 its depths drift away from the CPU's over a long run —
  locally by decimetres on A5 after hours — while the totals stay together.
  Two CPU runs with a different cell order differ in the same way. This is a
  property of the scheme, not of the device or the precision.
- **Precision.** Float32 carries about 7 digits. Over a few steps its
  depths differ from Float64 by 10⁻⁷–10⁻⁶ m; over a long run those
  differences are amplified exactly as round-off is in Float64.

So judge Float32 by whether its distance from the reference is **larger than
GPU Float64's**, and on quantities a flood model is used for — storage,
flooded extent — rather than by the largest pointwise difference. The plan of
record's acceptance values (`projectinstructions`, §7.3: max |Δh| < 10⁻⁴ m,
relative mass error < 10⁻⁶, extent CSI > 0.99) are met on short runs and on
the Cartesian mesh (`test/test_precision.jl`); on long A5 runs the first
cannot hold for any second run, Float64 included.

**After 20 steps** (the step benchmark of §3), against the CPU run of the
same steps:

In [ ]:
function step_accuracy_table(r)
    rws = BS.rows(r, :step); out = []
    for solver in ("standard", "swe"), n in sort(unique(x["n"] for x in sel(rws; solver)))
        g64 = pick(rws; solver, n, config = "gpu64"); g32 = pick(rws; solver, n, config = "gpu32")
        (g64 === nothing && g32 === nothing) && continue
        push!(out, [solver, "$(n) × $(n)", @sprintf("%.1e", val(g64, "max_dh")), @sprintf("%.1e", val(g32, "max_dh")),
                    @sprintf("%.1e", val(g32, "rms_dh")), @sprintf("%.1e", val(g64, "mass_rel")),
                    @sprintf("%.1e", val(g32, "mass_rel"))])
    end
    mdtable(["solver", "grid", "max |Δh| GPU Float64 (m)", "max |Δh| GPU Float32 (m)", "rms Δh GPU Float32 (m)",
             "mass error GPU Float64", "mass error GPU Float32"], out)
end
step_accuracy_table(mine)

**At the end of the Carlisle runs**, each against the first configuration of
its mesh and model (the CPU run; GPU Float64 where there is no CPU run):
storage ratio, critical success index of the extent deeper than 0.1 m, the
rms depth difference over wet cells, the median and largest depth difference
over all cells, the share of cells that differ by more than 1 cm, and each
run's own conservation error (volume stored and removed against the volume
the stepping added, relative).

In [ ]:
function carlisle_accuracy_table(r)
    rws = BS.rows(r, :carlisle); out = []
    for x in rws
        haskey(x, "ref_config") || continue
        push!(out, [x["mesh"], x["model"], CONFIG_LABEL[x["config"]] * " vs " * CONFIG_LABEL[x["ref_config"]],
                    f(x["t_compare_s"] / 3600, 2), @sprintf("%+.1e", x["storage_ratio"] - 1), f(x["csi"], 4),
                    @sprintf("%.1e", x["rms_dh_wet"]), @sprintf("%.1e", x["median_dh"]), @sprintf("%.1e", x["max_dh"]),
                    f(100 * x["frac_dh_gt_1cm"], 2) * "%", @sprintf("%.1e", x["conservation_rel"])])
    end
    mdtable(["mesh", "model", "comparison", "at (h)", "storage ratio − 1", "CSI (0.1 m)", "rms Δh wet (m)",
             "median |Δh| (m)", "max |Δh| (m)", "cells with |Δh| > 1 cm", "conservation error"], out)
end
carlisle_accuracy_table(mine)

## 7. The machines side by side

One line per machine: the step throughput on the 1024 × 1024 grid (about a
million cells) and the wall time of the 5 m Carlisle run (standard flow, one
simulated hour).

In [ ]:
function summary_table(results; n = 1024)
    out = []
    for r in results
        st = BS.rows(r, :step); ca = BS.rows(r, :carlisle)
        row = Any[name(r), r["machine"]["cpu"], r["machine"]["gpu"] == "" ? "—" : r["machine"]["gpu"]]
        for solver in ("standard", "swe"), config in CONFIGS
            push!(row, f(val(pick(st; solver, n, config), "mcs"), 1))
        end
        for config in CONFIGS
            push!(row, f(val(pick(ca; mesh = "cart5", model = "standard", config), "wall_s"), 0))
        end
        push!(out, row)
    end
    mdtable(["machine", "CPU", "GPU", "standard CPU (Mc·s/s)", "standard GPU64", "standard GPU32",
             "SWE CPU (Mc·s/s)", "SWE GPU64", "SWE GPU32", "Carlisle 5 m CPU (s)", "GPU64 (s)", "GPU32 (s)"], out)
end
summary_table(results)

## Notes

- **Reproducing a number.** Each results file records the FloodA5 version and
  commit of every section, and each Carlisle row its command. The step rows
  are `julia --project=. -t <threads> test/benchmark_backend.jl --solver
  <solver> --sizes <n> --configs cpu,gpu64,gpu32`.
- **What is not covered.** SGS (`--flow-model sgs`) has no Float32 path yet.
  GPUs other than NVIDIA/CUDA are not supported (the kernels are written
  against KernelAbstractions.jl, so AMD, Intel and Apple back ends are a
  matter of loading another package, but none has been tested). Multi-GPU
  runs are not supported.
- **Reading a results file elsewhere.** `benchmarks/results/*.json` is plain
  JSON (`benchmarks/README.md` describes the fields).